Import some libraries first.

Notice the `%%local` tag. By default all code is **executed remotely** as PySpark, so you need to annotate cells with `%%local` if you want them to run in your local Python kernel.

We will do all data processing with SQL and only get data to draw a chart, so we only need a few plotting libraries.

In [ ]:
%%local
import pandas as pd
import hvplot.pandas

Next we initialize `kamu` extension and import two datasets, giving them shorter aliases

In [ ]:
%load_ext kamu

In [ ]:
%import_dataset weather-station-001 --alias station1
%import_dataset weather-station-002 --alias station2

The `%%sql` cells let you to execute SQL remotely and quickly view the results.

Try switching to Line / Area / Bar plots and selecting x / y axis.

Notice that `-o df` parameter makes SQL results available to the local kernel in `df` variable as Pandas Dataframe.

In [ ]:
%%sql -o df

-- Calculates discrete derivative to produce an hourly precipitation rate

select
    device_id,
    date_trunc("HOUR", event_time) as event_time,
    max(precipitation_accumulated) - min(precipitation_accumulated) as precip_rate
from (
    select "001" as device_id, * from station1
    union all
    select "002" as device_id, * from station2
)
group by 1, 2

In [ ]:
%%local

df.hvplot.line(
    x="event_time",
    y="precip_rate",
    by="device_id",
    xlabel="Time",
    ylabel="Rate (mm / h)",
    width=900,
    height=500,
    legend="top_right"
)